# 15

In [5]:
import pandas as pd
from pathlib import Path

current_dir = Path.cwd()
data_path = current_dir / "sampled_reviews.csv"

print("current_dir:", current_dir)
print("data_path:", data_path)

current_dir: /Users/rebecca/Documents/CROUSE/Winter-2025/COMP262-NLP_RS/Project/W25_COMP262_002_TeamGamma
data_path: /Users/rebecca/Documents/CROUSE/Winter-2025/COMP262-NLP_RS/Project/W25_COMP262_002_TeamGamma/sampled_reviews.csv


In [ ]:
#load data

df = pd.read_csv(data_path)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   overall                  2000 non-null   int64  
 1   verified                 2000 non-null   bool   
 2   reviewTime               2000 non-null   object 
 3   reviewerID               2000 non-null   object 
 4   asin                     2000 non-null   object 
 5   reviewerName             2000 non-null   object 
 6   reviewText               2000 non-null   object 
 7   summary                  2000 non-null   object 
 8   unixReviewTime           2000 non-null   int64  
 9   vote                     189 non-null    float64
 10  style                    801 non-null    object 
 11  image                    28 non-null     object 
 12  reviewText_length_words  2000 non-null   int64  
 13  label                    2000 non-null   object 
dtypes: bool(1), float64(1), 

In [ ]:
df.head(5)

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image,reviewText_length_words,label
0,5,True,"06 5, 2014",A3H1T313YSK2U0,B00AZBIZTW,maria abreu,This is a great vacuum! I have gone through a ...,GREAT PRODUCT,1401926400,NaN,"{'Color:': ' Purple', 'Style Name:': ' Vacuum ...",NaN,37,POSITIVE
1,3,True,"11 22, 2014",A24FAUV92Z4NQ2,B0018MKX5Q,Roaster,"The clamp is nice and strong, but the one I go...",Perpendiculous screw clamp,1416614400,NaN,NaN,NaN,50,NEUTRAL
2,5,True,"10 9, 2016",A13QDUSC20Z4E3,B00C351GBC,Andria,For now we are vacuuming a lot more because of...,Love to vacuum,1475971200,NaN,{'Color:': ' White'},NaN,35,POSITIVE
3,3,True,"12 19, 2016",A1PWHZ9C7U3AGF,B00AFCEJX2,Matthew M.,The buttons work as expected. Packaging was b...,good buttons,1482105600,NaN,NaN,NaN,37,NEUTRAL
4,5,True,"02 18, 2015",A5A4RGSDKK124,B005W0UWCY,t.allman,Easy to install. These lifted up my 500b mapl...,Easy to install. Worked great.,1424217600,NaN,NaN,NaN,40,POSITIVE


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from sklearn.preprocessing import LabelEncoder


df["text"] = df["summary"] + " " + df["reviewText"]

df["user_item"] = df["reviewerID"] + "_" + df["asin"]

user_encoder = LabelEncoder()
item_encoder = LabelEncoder()

df["user_id"] = user_encoder.fit_transform(df["reviewerID"])
df["item_id"] = item_encoder.fit_transform(df["asin"])

In [ ]:
# Sentiment score
df["sentiment"] = df["text"].apply(lambda x: TextBlob(x).sentiment.polarity)


In [ ]:
# TF-IDF similarity
tfidf_vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df["text"])
tfidf_sim = cosine_similarity(tfidf_matrix)


In [ ]:
# Hybrid similarity
hybrid_sim = tfidf_sim

def hybrid_score(index, top_n=5):
    scores = list(enumerate(hybrid_sim[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    indices = [i for i, _ in scores]
    avg_rating = df.iloc[indices]["overall"].mean()
    avg_sentiment = df.iloc[indices]["sentiment"].mean()
    return 0.6 * avg_rating + 0.4 * (avg_sentiment * 5)

df["enhanced_rating"] = [
    0.5 * df.iloc[i]["overall"] + 0.5 * hybrid_score(i)
    for i in range(len(df))
]

df[["summary", "reviewText", "overall", "sentiment", "enhanced_rating"]].to_csv("enhanced_output_advanced.csv", index=False)

def recommend_items(index, top_n=5):
    scores = list(enumerate(hybrid_sim[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    indices = [i for i, _ in scores]
    recommendations = df.iloc[indices][["summary", "reviewText", "overall", "enhanced_rating"]]
    print(f"Top {top_n} recommendations for item {index}:")
    for i, row in recommendations.iterrows():
        print(f"\n--- Recommendation {i} ---")
        print(f"Summary: {row['summary']}")
        print(f"Review: {row['reviewText']}")
        print(f"Enhanced Rating: {row['enhanced_rating']:.2f}")

recommend_items(0)

In [ ]:
def generate_html_recommendations(index, top_n=5, output_file="recommendations.html"):
    scores = list(enumerate(hybrid_sim[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    indices = [i for i, _ in scores]
    recommendations = df.iloc[indices][["summary", "reviewText", "overall", "enhanced_rating"]]
    
    html = f"""
    <html>
    <head>
        <title>Top {top_n} Recommendations for Item {index}</title>
        <style>
            body {{ font-family: Arial, sans-serif; padding: 20px; }}
            .rec {{ border: 1px solid #ddd; padding: 15px; margin-bottom: 10px; border-radius: 8px; }}
            .rating {{ color: #f39c12; font-weight: bold; }}
        </style>
    </head>
    <body>
        <h1>Top {top_n} Recommendations for Item {index}</h1>
    """

    for _, row in recommendations.iterrows():
        html += f"""
        <div class="rec">
            <h2>{row['summary']}</h2>
            <p>{row['reviewText']}</p>
            <p class="rating">Original Rating: {row['overall']:.2f}</p>
            <p class="rating">Enhanced Rating: {row['enhanced_rating']:.2f}</p>
        </div>
        """

    html += "</body></html>"

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Recommendations saved to {output_file}")

generate_html_recommendations(0)

In [ ]:
import matplotlib.pyplot as plt  

plt.figure(figsize=(10, 5))
plt.hist(df['overall'], bins=5, alpha=0.5, label='Original')
plt.hist(df['enhanced_rating'], bins=5, alpha=0.5, label='Enhanced')
plt.title('Original vs Enhanced Ratings')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.legend()
plt.show()